<div style="
  border:3px solid #003366;      /* azul oscuro (borde) */
  border-radius:10px;
  background-color:#CFE8FF;      /* azul claro (fondo) */
  padding:4px 8px;              /* menos espacio interno */
  overflow:auto;">

  <!-- logo sin cambios -->
  <img src="data/imgs/logo.png" align="right" width="200" alt="Logo">

  <!-- Datathon sin cambios -->
  <font color="#1874CD"><h1 align="left" style="margin-top:0;"><b>Scrapping de YouTube</b></h1></font>

  <!-- Resto en azul oscuro -->
  <h2 align="left" style="color:#003366;margin:0;"><b>Jeremy Aguilar Marin</b></h2>
  <h3 align="left" style="color:#003366;margin:0;">Proyecto Practicas Curriculares ROI</h3>
</div>

<hr>

<div style="
  border:3px solid #003366;      /* azul oscuro (borde) */
  border-radius:10px;
  background-color:rgba(207,232,255,0.6);      /* azul claro (fondo) */
  padding:4px 8px;              /* menos espacio interno */
  overflow:auto;">
<h2 align="left" style="color:#003366;margin:0;"><b>Librerías</b></h2>
</div>

In [42]:
#Para extraer los comentarios en YouTube sin todo el scrapeo 
#pip install youtube-comment-downloader

#Analisis de HTML
#pip install  beautifulsoup4

#Para visualizacion
#pip install matplotlib seaborn

# Análisis de cometarios
#pip install scikit-learn nltk



import requests
import pandas as pd
import numpy as np

import yt_dlp
from youtube_comment_downloader import YoutubeCommentDownloader

import matplotlib.pyplot as plt
import seaborn as sns

from bs4 import BeautifulSoup

from sklearn.feature_extraction.text import TfidfVectorizer
import nltk


<div style="
  border:3px solid #003366;      /* azul oscuro (borde) */
  border-radius:10px;
  background-color:rgba(207,232,255,0.6);      /* azul claro (fondo) */
  padding:4px 8px;              /* menos espacio interno */
  overflow:auto;">
<h2 align="left" style="color:#003366;margin:0;"><b>Links</b></h2>
</div>

In [43]:
url = "https://www.youtube.com/watch?v=pS6_nWlZmvA"

<div style="
  border:3px solid #003366;      /* azul oscuro (borde) */
  border-radius:10px;
  background-color:rgba(207,232,255,0.6);      /* azul claro (fondo) */
  padding:4px 8px;              /* menos espacio interno */
  overflow:auto;">
<h2 align="left" style="color:#003366;margin:0;"><b>Librería yt_dlp</b></h2>
</div>

In [44]:
# Configuración de la libreria 
ydl_opts = {
    "quiet": True,
    "cookiesfrombrowser": ("firefox",) # Cookies de Firefox para acceder a YouTube
}

<div style="
  border:3px solid #003366;      /* azul oscuro (borde) */
  border-radius:10px;
  background-color:rgba(207,232,255,0.6);      /* azul claro (fondo) */
  padding:4px 8px;              /* menos espacio interno */
  overflow:auto;">
<h2 align="left" style="color:#003366;margin:0;"><b>Extracción de Metadatos</b></h2>
</div>

In [39]:
# Extracción de los datos de YouTube
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(url, download=False)

# Datos buscados
datos = {
    "ID": info.get("id"), 
    "TITULO": info.get("title"),  
    "CANAL": info.get("channel"),  
    "FECHA": info.get("upload_date"),  
    "VIEWS": info.get("view_count"),  
    "LIKES": info.get("like_count"),  
    "DURACION": info.get("duration"),  
    "DESCRIPCION": info.get("description"),  
    "URL": info.get("webpage_url")  
}

<div style="
  border:3px solid #003366;      /* azul oscuro (borde) */
  border-radius:10px;
  background-color:rgba(207,232,255,0.6);      /* azul claro (fondo) */
  padding:4px 8px;              /* menos espacio interno */
  overflow:auto;">
<h2 align="left" style="color:#003366;margin:0;"><b>DataFrame</b></h2>
</div>

In [40]:
df = pd.DataFrame([datos])
df

,ID,TITULO,CANAL,FECHA,VIEWS,LIKES,DURACION,DESCRIPCION,URL
0,pS6_nWlZmvA,Las tres jugadas sucias de Sánchez para ganar ...,Juan Ramón Rallo,20261006,366342,10641,1211,🗳️ ¿Qué está dispuesto a hacer Pedro Sánchez p...,https://www.youtube.com/watch?v=pS6_nWlZmvA


<div style="
  border:3px solid #003366;      /* azul oscuro (borde) */
  border-radius:10px;
  background-color:rgba(207,232,255,0.6);      /* azul claro (fondo) */
  padding:4px 8px;              /* menos espacio interno */
  overflow:auto;">
<h2 align="left" style="color:#003366;margin:0;"><b>Comentarios</b></h2>
</div>

In [41]:
downloader = YoutubeCommentDownloader()

comentarios = list(downloader.get_comments_from_url(url))

# Mostrar el número de comentarios obtenidos
len(comentarios)

1159

<div style="
  border:3px solid #003366;      /* azul oscuro (borde) */
  border-radius:10px;
  background-color:rgba(207,232,255,0.6);      /* azul claro (fondo) */
  padding:4px 8px;              /* menos espacio interno */
  overflow:auto;">
<h2 align="left" style="color:#003366;margin:0;"><b>Dataset comentarios</b></h2>
</div>

In [45]:
# Lista de los comentarios
lista_comentarios = []

for comentario in comentarios:

    # Datos buscados
    datos_comentario = {
        "ID_COMENTARIO": comentario.get("cid"),  
        "ID_VIDEO": info.get("id"),  
        "TITULO_VIDEO": info.get("title"),  
        "COMENTARIO": comentario.get("text"),  
        "AUTOR": comentario.get("author"),  
        "LIKES": comentario.get("votes"),  
        "RESPUESTAS": comentario.get("replies"),  
        "FECHA": comentario.get("time"),  
        "CORAZON": comentario.get("heart"),  # Si al Youtuber le dió like al comentario
        "RESPUESTA": comentario.get("reply")  # Si es una respuesta a otro comentario
    }

  
    lista_comentarios.append(datos_comentario)

<div style="
  border:3px solid #003366;      /* azul oscuro (borde) */
  border-radius:10px;
  background-color:rgba(207,232,255,0.6);      /* azul claro (fondo) */
  padding:4px 8px;              /* menos espacio interno */
  overflow:auto;">
<h2 align="left" style="color:#003366;margin:0;"><b> DataFrame comentarios</b></h2>
</div>

In [46]:
df_c = pd.DataFrame(lista_comentarios)
df_c.head()


,ID_COMENTARIO,ID_VIDEO,TITULO_VIDEO,COMENTARIO,AUTOR,LIKES,RESPUESTAS,FECHA,CORAZON,RESPUESTA
0,UgxN6zIOB0MTThGnszB4AaABAg,pS6_nWlZmvA,Las tres jugadas sucias de Sánchez para ganar ...,La única jugada del PSOE es que se ha movido d...,@pedrogonzalez-it7bo,0,,hace 2 horas,False,False
1,UgwEBiBp0Us2U7XY9nR4AaABAg,pS6_nWlZmvA,Las tres jugadas sucias de Sánchez para ganar ...,"Da igual, por Ley hay una medida cautelar",@jorgecastillejo2338,0,,hace 3 horas,False,False
2,UgwD6jtpRpCVgmePr6h4AaABAg,pS6_nWlZmvA,Las tres jugadas sucias de Sánchez para ganar ...,Y el PNV va a aprobar los decretos MariCarmen ...,@jorgecastillejo2338,0,,hace 3 horas,False,False
3,UgweTjOrfvmV9hGopBp4AaABAg,pS6_nWlZmvA,Las tres jugadas sucias de Sánchez para ganar ...,Si gana las elecciones no es por jugadas es po...,@Nanao714,0,,hace 4 horas,False,False
4,UgwGjNEtl2WJh_kkRBF4AaABAg,pS6_nWlZmvA,Las tres jugadas sucias de Sánchez para ganar ...,La pena es usar a una mujer que la han desahuc...,@joseantoniovallejo9763,0,,hace 4 horas,False,False


In [ ]:
df_c.isnull().sum()

ID_COMENTARIO    0
ID_VIDEO         0
TITULO_VIDEO     0
COMENTARIO       0
AUTOR            0
LIKES            0
RESPUESTAS       0
FECHA            0
CORAZON          0
RESPUESTA        0
dtype: int64

In [49]:
df_c["ID_COMENTARIO"].duplicated().sum()

np.int64(0)

In [50]:
df_c.shape

(1159, 10)

In [52]:
df_c.dtypes

ID_COMENTARIO     str
ID_VIDEO          str
TITULO_VIDEO      str
COMENTARIO        str
AUTOR             str
LIKES             str
RESPUESTAS        str
FECHA             str
CORAZON          bool
RESPUESTA        bool
dtype: object

In [53]:
df_c["LIKES"].head(10)

0    0
1    0
2    0
3    0
4    0
5    0
6    0
7    0
8    0
9    0
Name: LIKES, dtype: str

In [54]:
df_c["RESPUESTAS"].head(10)

0    
1    
2    
3    
4    
5    
6    
7    
8    
9    
Name: RESPUESTAS, dtype: str

In [55]:
df_c["LIKES"].value_counts().head(20)

LIKES
0     726
1     203
2      53
3      31
7      19
4      18
6      17
5      12
8      11
9       8
10      7
11      6
13      3
80      3
14      3
23      3
30      2
16      2
25      2
20      2
Name: count, dtype: int64

In [56]:
df_c["RESPUESTAS"].value_counts().head(20)

RESPUESTAS
      1082
1       33
2       16
3        8
8        4
7        2
10       2
14       2
4        1
6        1
11       1
12       1
36       1
26       1
66       1
16       1
5        1
49       1
Name: count, dtype: int64